# 03 - Stock Market Analysis 📈

Let's use pandas and matplotlib on **real** data: five years of daily stock prices (Sept 2021 – Aug 2026) from Yahoo Finance, saved in `data/`.

| Ticker | Company / fund |
|--------|----------------|
| AAPL | Apple |
| MSFT | Microsoft |
| NVDA | NVIDIA |
| KO | Coca-Cola |
| SPY | An index fund that tracks the S&P 500 (500 large US companies, "the market") |

Each row is one trading day:
- **Open**: price at 9:30am
- **High / Low**: the day's highest and lowest price
- **Close**: price at 4pm
- **Volume**: number of shares traded

Prices are *adjusted* for stock splits and dividends, so they're comparable over time.

> ⚠️ This is a data-analysis exercise, **not financial advice**. Past performance doesn't predict future results.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

aapl = pd.read_csv("data/AAPL.csv", parse_dates=["Date"], index_col="Date")   # use dates as the row labels
aapl.head()

In [ ]:
print(aapl.shape)
print("From", aapl.index.min().date(), "to", aapl.index.max().date())
aapl.describe().round(2)

## 1. The price over time

In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(aapl["Close"])
plt.title("AAPL closing price")
plt.ylabel("USD")
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# Selecting by date is easy when dates are the index
aapl.loc["2025-04-01":"2025-04-15", ["Close", "Volume"]]

## 2. Daily returns: how much did the price change each day?
**return** = (today − yesterday) / yesterday. pandas does it in one call: `.pct_change()`.

In [ ]:
aapl["Return"] = aapl["Close"].pct_change()
aapl[["Close", "Return"]].head()

In [ ]:
print(f"Average daily return: {aapl['Return'].mean():.3%}")
print(f"Best day:  {aapl['Return'].idxmax().date()}  {aapl['Return'].max():+.2%}")
print(f"Worst day: {aapl['Return'].idxmin().date()}  {aapl['Return'].min():+.2%}")
print(f"Up days: {(aapl['Return'] > 0).mean():.1%} of the time")

The best and worst days were a week apart in **April 2025**, when new tariff announcements shook the whole market. Big news moves prices, and news is hard to predict.

In [ ]:
plt.hist(aapl["Return"].dropna(), bins=80, edgecolor="black")
plt.xlabel("Daily return")
plt.ylabel("Number of days")
plt.title("Most days are small moves; a few are huge")
plt.show()

## 3. Moving averages: smoothing out the noise
A 50-day moving average is the average of the last 50 closing prices. `.rolling(50).mean()` computes it for every day.

In [ ]:
aapl["MA50"] = aapl["Close"].rolling(50).mean()
aapl["MA200"] = aapl["Close"].rolling(200).mean()

plt.figure(figsize=(12, 5))
plt.plot(aapl["Close"], label="Close", alpha=0.5)
plt.plot(aapl["MA50"], label="50-day average")
plt.plot(aapl["MA200"], label="200-day average")
plt.legend()
plt.title("AAPL with moving averages")
plt.show()

## 4. Comparing stocks: normalize to a common start
The raw prices aren't comparable ($100 vs $500 per share). Instead, divide by the first price and multiply by 100: "if it started at 100, where is it now?"

In [ ]:
tickers = ["AAPL", "MSFT", "NVDA", "KO", "SPY"]
closes = pd.DataFrame({t: pd.read_csv(f"data/{t}.csv", parse_dates=["Date"], index_col="Date")["Close"]
                       for t in tickers})
closes.head()

In [ ]:
normalized = closes / closes.iloc[0] * 100

normalized.plot(figsize=(12, 5), title="Growth of each stock (start = 100)", logy=True)
plt.ylabel("Value (log scale)")
plt.axhline(100, color="gray", linestyle="--")
plt.show()

In [ ]:
# What would $1,000 invested on day one be worth at the end?
final_values = (1000 * closes.iloc[-1] / closes.iloc[0]).round(2).sort_values(ascending=False)
final_values

## 5. Risk: volatility
**Volatility** = the standard deviation of daily returns, i.e. how bumpy the ride is. Multiply by √252 (about 252 trading days per year) to annualize it.

In [ ]:
returns = closes.pct_change().dropna()
summary = pd.DataFrame({
    "avg yearly return": returns.mean() * 252,
    "volatility": returns.std() * np.sqrt(252),
}).round(3)
summary

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(summary["volatility"], summary["avg yearly return"], s=100)
for t in summary.index:
    plt.annotate(t, (summary.loc[t, "volatility"], summary.loc[t, "avg yearly return"]),
                 xytext=(6, 6), textcoords="offset points")
plt.xlabel("Volatility (risk)")
plt.ylabel("Average yearly return")
plt.title("Risk vs return")
plt.grid(alpha=0.3)
plt.show()

**Takeaway:** NVDA had by far the highest return, and also by far the bumpiest ride. KO and SPY were the calmest, yet over these particular five years they grew about as much as MSFT. Risk and reward are related, but not perfectly, and five years is a small sample.

## 6. Do stocks move together? Correlation
+1 means "always move the same direction," 0 means "unrelated," and −1 means "opposite directions."

In [ ]:
returns.corr().round(2)

## 7. (Optional) Live data
With internet access and `pip install yfinance`, you can download any ticker:

In [ ]:
try:
    import yfinance as yf
    live = yf.download("GOOGL", period="6mo", progress=False, auto_adjust=True)
    live.columns = live.columns.get_level_values(0)     # flatten yfinance's 2-level column names
    print(live.tail())
except Exception as e:
    print("Live download not available here:", type(e).__name__)

## ✏️ Try it
1. Which of the five had the most "up" days (as a percentage)?
2. Plot MSFT's close with its 20-day moving average for just the year **2024**. (Hint: `.loc["2024"]`.)
3. For NVDA, find the **month** with the biggest total gain. (Hint: `closes["NVDA"].groupby(closes.index.to_period("M")).last().pct_change()`)
4. What was the biggest drop from a peak ("max drawdown") for SPY? (Hint: `closes["SPY"] / closes["SPY"].cummax() - 1`, then `.min()`.)

In [ ]:
# your code here